# Q3 — Audio Entropy Analysis (Raw)
#
# Pipeline:
#   (1) Load raw audio at 16 kHz mono
#   (2) Remove DC offset
#   (3) Temporal trim: skip first 0.5 s, then take 4 s → all clips are exactly 4 s
#   (4) Compute entropy metrics (no peak normalization)

In [ ]:
import numpy as np
import pandas as pd
import librosa
import os

PATH_AMBIENT = 'data/Q3_Ambient'
PATH_MUSIC   = 'data/Q3_Music'

TARGET_SR = 16000
CLIP_DURATION = 4.0  # seconds
CLIP_SAMPLES = int(TARGET_SR * CLIP_DURATION)  # 64000 samples
TRIM_SECONDS = 0.5

def load_clip(path, sr=TARGET_SR, trim_seconds=TRIM_SECONDS, duration=CLIP_SAMPLES):
    """Load audio, skip first 0.5s, take 4s, remove DC offset."""
    audio, _ = librosa.load(path, sr=sr, mono=True)
    # Skip first 0.5s (consistent with temporal trimming in preprocessing)
    trim_samples = int(trim_seconds * sr)
    audio = audio[trim_samples:]
    # Take first 4s
    audio = audio[:duration]
    # Remove DC offset
    audio = audio - np.mean(audio)
    return audio

# Verify paths
for label, path in [('Ambient', PATH_AMBIENT), ('Music', PATH_MUSIC)]:
    files = sorted([f for f in os.listdir(path) if f.endswith(('.wav', '.mp3'))])
    print(f'{label}: {len(files)} files in {path}')

In [ ]:
# Load all clips and concatenate for global bin width
all_audio = []
clip_data = {}  # {class: [audio_array, ...]}

for label, path in [('Ambient', PATH_AMBIENT), ('Music', PATH_MUSIC)]:
    files = sorted([f for f in os.listdir(path) if f.endswith(('.wav', '.mp3'))])
    clips = []
    for f in files:
        audio = load_clip(os.path.join(path, f))
        clips.append(audio)
        all_audio.append(audio)
    clip_data[label] = clips
    print(f'{label}: loaded {len(clips)} clips, each {CLIP_SAMPLES} samples')

all_audio = np.concatenate(all_audio)
n_total = len(all_audio)

# Freedman-Diaconis rule
q75, q25 = np.percentile(all_audio, [75, 25])
iqr = q75 - q25
delta = 2 * iqr / n_total ** (1/3)
bin_edges = np.arange(-1, 1 + delta, delta)
K = len(bin_edges) - 1
H_max = np.log2(K)

print(f'\nTotal samples: {n_total:,}')
print(f'IQR: {iqr:.6f}')
print(f'Bin width (delta): {delta:.6f}')
print(f'Number of bins K: {K}')
print(f'H_max = log2(K): {H_max:.4f}')

In [ ]:
from scipy.stats import shapiro

results = []
for label in ['Ambient', 'Music']:
    for i, audio in enumerate(clip_data[label], 1):
        counts, _ = np.histogram(audio, bins=bin_edges)
        p = counts / counts.sum()
        p = p[p > 0]

        H_Q = -np.sum(p * np.log2(p))
        h_hat = H_Q + np.log2(delta)
        sigma2 = np.var(audio)
        H_gauss = 0.5 * np.log2(2 * np.pi * np.e * sigma2) if sigma2 > 0 else 0
        gap = H_gauss - h_hat

        # Shapiro-Wilk W (subsample to 5000 for scipy limit)
        rng = np.random.default_rng(42)
        sub = rng.choice(audio, size=5000, replace=False)
        W, _ = shapiro(sub)

        results.append({
            'Class': label,
            'Clip': i,
            'H_Q': round(H_Q, 3),
            'h_hat': round(h_hat, 3),
            'H_max': round(H_max, 3),
            'H_gauss': round(H_gauss, 3),
            'gap': round(gap, 3),
            'W': round(W, 4),
        })

df = pd.DataFrame(results)

# === Two separate tables ===
for label in ['Ambient', 'Music']:
    sub = df[df['Class'] == label][['Clip', 'H_Q', 'h_hat', 'H_max', 'H_gauss', 'gap', 'W']].copy()
    sub = sub.reset_index(drop=True)
    print(f'\n{"="*60}')
    print(f'  {label}')
    print(f'{"="*60}')
    print(sub.to_string(index=False))

In [ ]:
# Summary statistics per class
summary = df.groupby('Class')[['H_Q', 'h_hat', 'H_gauss', 'gap', 'W']].agg(['mean', 'std'])
print('=== Summary Statistics ===')
display(summary.round(3))

In [ ]:
import matplotlib
matplotlib.use('Agg')
import os
os.makedirs('3', exist_ok=True)
# Gap cloud plot
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(8, 5))
colors = {'Ambient': '#1f77b4', 'Music': '#2ca02c'}
for i, label in enumerate(['Ambient', 'Music']):
    sub = df[df['Class'] == label]
    gaps = sub['gap'].values
    jitter = np.random.default_rng(0).uniform(-0.2, 0.2, len(gaps))
    ax.scatter([i]*len(gaps) + jitter, gaps, color=colors[label], s=60, alpha=0.7, zorder=3)
    mean_gap = gaps.mean()
    ax.scatter([i], [mean_gap], color=colors[label], marker='D', s=120,
               edgecolor='black', linewidth=1.5, zorder=4)

ax.set_xticks([0, 1])
ax.set_xticklabels(['Ambient', 'Music'], fontsize=16)
ax.set_ylabel('Gap ($H_{Gauss} - \\hat{H}$)', fontsize=18)
ax.set_title('Entropy Gap by Class (diamond = mean)', fontsize=18)
ax.tick_params(axis='y', labelsize=14)
ax.legend(fontsize=12)
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.savefig('3/gap_cloud_raw.png', dpi=150, bbox_inches='tight')
plt.close()
print("Saved 3/gap_cloud_raw.png")

---

# Q3 Normalized

Same pipeline but with **peak normalization** to [-1, 1] after DC removal.

## Normalized Analysis
#
# Pipeline:
#   (1) Load raw audio at 16 kHz mono
#   (2) Remove DC offset
#   (3) Temporal trim: skip first 0.5 s, then take 4 s → all clips are exactly 4 s
#   (4) Peak-normalize to [-1, 1]
#   (5) Compute entropy metrics

In [ ]:
import numpy as np
import pandas as pd
import librosa
import os

PATH_AMBIENT = 'data/Q3_Ambient'
PATH_MUSIC   = 'data/Q3_Music'

TARGET_SR = 16000
CLIP_DURATION = 4.0  # seconds
CLIP_SAMPLES = int(TARGET_SR * CLIP_DURATION)  # 64000 samples
TRIM_SECONDS = 0.5

def load_clip(path, sr=TARGET_SR, trim_seconds=TRIM_SECONDS, duration=CLIP_SAMPLES):
    """Load audio, skip first 0.5s, take 4s, remove DC, peak-normalize to [-1,1]."""
    audio, _ = librosa.load(path, sr=sr, mono=True)
    # Skip first 0.5s (consistent with temporal trimming in preprocessing)
    trim_samples = int(trim_seconds * sr)
    audio = audio[trim_samples:]
    # Take first 4s
    audio = audio[:duration]
    # Remove DC offset
    audio = audio - np.mean(audio)
    # Peak normalize
    peak = np.max(np.abs(audio))
    if peak > 0:
        audio = audio / peak
    return audio

# Verify paths
for label, path in [('Ambient', PATH_AMBIENT), ('Music', PATH_MUSIC)]:
    files = sorted([f for f in os.listdir(path) if f.endswith(('.wav', '.mp3'))])
    print(f'{label}: {len(files)} files in {path}')

In [ ]:
# Load all clips and concatenate for global bin width
all_audio = []
clip_data = {}  # {class: [audio_array, ...]}

for label, path in [('Ambient', PATH_AMBIENT), ('Music', PATH_MUSIC)]:
    files = sorted([f for f in os.listdir(path) if f.endswith(('.wav', '.mp3'))])
    clips = []
    for f in files:
        audio = load_clip(os.path.join(path, f))
        clips.append(audio)
        all_audio.append(audio)
    clip_data[label] = clips
    print(f'{label}: loaded {len(clips)} clips, each {CLIP_SAMPLES} samples')

all_audio = np.concatenate(all_audio)
n_total = len(all_audio)

# Freedman-Diaconis rule
q75, q25 = np.percentile(all_audio, [75, 25])
iqr = q75 - q25
delta = 2 * iqr / n_total ** (1/3)
K = int(np.ceil((2.0) / delta))  # bins over [-1, 1]
H_max = np.log2(K)

print(f'\nTotal samples: {n_total:,}')
print(f'IQR: {iqr:.6f}')
print(f'Bin width (delta): {delta:.6f}')
print(f'Number of bins K: {K}')
print(f'H_max = log2(K): {H_max:.4f}')

In [ ]:
from scipy.stats import shapiro

bin_edges = np.arange(-1, 1 + delta, delta)

results = []
for label in ['Ambient', 'Music']:
    for i, audio in enumerate(clip_data[label], 1):
        counts, _ = np.histogram(audio, bins=bin_edges)
        p = counts / counts.sum()
        p = p[p > 0]

        H_Q = -np.sum(p * np.log2(p))
        h_hat = H_Q + np.log2(delta)
        sigma2 = np.var(audio)
        H_gauss = 0.5 * np.log2(2 * np.pi * np.e * sigma2) if sigma2 > 0 else 0
        gap = H_gauss - h_hat

        # Shapiro-Wilk W (subsample to 5000 for scipy limit)
        rng = np.random.default_rng(42)
        sub = rng.choice(audio, size=5000, replace=False)
        W, _ = shapiro(sub)

        results.append({
            'Class': label,
            'Clip': i,
            'H_Q': round(H_Q, 3),
            'h_hat': round(h_hat, 3),
            'H_max': round(H_max, 3),
            'H_gauss': round(H_gauss, 3),
            'gap': round(gap, 3),
            'W': round(W, 4),
        })

df = pd.DataFrame(results)

# === Two separate tables ===
for label in ['Ambient', 'Music']:
    sub = df[df['Class'] == label][['Clip', 'H_Q', 'h_hat', 'H_max', 'H_gauss', 'gap', 'W']].copy()
    sub = sub.reset_index(drop=True)
    print(f'\n{"="*60}')
    print(f'  {label}')
    print(f'{"="*60}')
    print(sub.to_string(index=False))

In [ ]:
# Summary statistics per class
summary = df.groupby('Class')[['H_Q', 'h_hat', 'H_gauss', 'gap', 'W']].agg(['mean', 'std'])
print('=== Summary Statistics ===')
display(summary.round(3))